# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [9]:
# import the requests library (1 line)
import requests
# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com"
# assign the /status endpoint to another variable called status_url (1 line)
status_url = "https://country-leaders.onrender.com/status"
# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(status_url)
# check the status_code using a condition and print appropriate messages (4 lines)
print(req, req.status_code)

ModuleNotFoundError: No module named 'requests'

### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [ ]:
import json
# Set the countries_url variable (1 line)
countries_url = "https://country-leaders.onrender.com/countries"
# query the /countries endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(countries_url)
# Get the JSON content and store it in the countries variable (1 line)
countries = req.json()
# display the request's status code and the countries variable (1 line)
print(req.status_code, countries.keys())


403 dict_keys(['message'])


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [ ]:
# Set the cookie_url variable (1 line)
cookie_url = "https://country-leaders.onrender.com/cookie"
# Query the enpoint, set the cookies variable and display it (2 lines)
cookies = requests.get(cookie_url)
print(cookies.cookies)

<RequestsCookieJar[<Cookie user_cookie=0b008925-fad1-4672-a2ff-e574d5dce1c0 for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [ ]:
# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(countries_url, cookies = cookies.cookies).json()
# display the countries variable (1 line)
print(countries)

['fr', 'us', 'be', 'ma', 'ru']


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [ ]:
# Set the leaders_url variable (1 line)
leaders_url = "https://country-leaders.onrender.com/leaders"
# query the /leaders endpoint, assign the output to the leaders variable (1 line)
leaders = requests.get(leaders_url, cookies= cookies.cookies)
# display the leaders variable (1 line)
print(leaders.json())

{'message': 'Please specify a country'}


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [ ]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
#leaders_be_url = "https://country-leaders.onrender.com/leaders?country=be"
# assign the output to the leaders variable (1 line)
#leaders= requests.get(leaders_be_url, cookies= cookies.cookies)
# display the leaders variable (1 line)
#print(leaders.json())

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [ ]:
# 4 lines
leaders_per_country = {}
for country in countries: 
    #print(type(country))
    leaders_url = "https://country-leaders.onrender.com/leaders?country=" + str(country)
    leaders = requests.get(leaders_url, cookies= cookies.cookies).json()
    leaders_per_country[str(country)] = leaders


print(countries)
print(len(countries))
print(type(countries[0]))


['fr', 'us', 'be', 'ma', 'ru']
5
<class 'str'>


In [ ]:
# or 1 line
print(len(leaders_per_country["be"]))

53


It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [ ]:
# < 15 lines
import requests
import json

def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    countries_url = "https://country-leaders.onrender.com/countries"
    cookies = requests.get(cookie_url)
    countries = requests.get(countries_url, cookies = cookies.cookies).json()
    leaders_per_country = {}
    for country in countries:
        leaders_url = "https://country-leaders.onrender.com/leaders?country=" + str(country)
        leaders = requests.get(leaders_url, cookies= cookies.cookies).json()
        leaders_per_country[str(country)] = leaders
    return leaders_per_country

Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [ ]:
# 2 lines
print(len(leaders_per_country))
print(leaders_per_country["us"][0])

5
{'id': 'Q23', 'first_name': 'George', 'last_name': 'Washington', 'birth_date': '1732-02-22', 'death_date': '1799-12-14', 'place_of_birth': 'Westmoreland County', 'wikipedia_url': 'https://en.wikipedia.org/wiki/George_Washington', 'start_mandate': '1789-04-30', 'end_mandate': '1797-03-04'}


## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [ ]:
# 3 lines
headers ={
    "User-Agent": "Mozilla/5.0"
}
url = "https://nl.wikipedia.org/wiki/Guy_Verhofstadt"
response = requests.get(url, headers = headers)
print(response.text[:500])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [ ]:
# 3 lines
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
print(soup.prettify())

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="nl">
 <head>
  <meta charset="utf-8"/>
  <title>
   Guy Verhofstadt - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-p

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [ ]:
# 2 lines

paragraphs = soup.find_all("p")
print(paragraphs)


[<p id="mwEw"><b id="mwFA">Guy Maurice Marie-Louise Verhofstadt</b><sup about="#mwt8" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-5"}}' id="cite_ref-5" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-5" id="mwFQ"><span class="mw-reflink-text" id="mwFg"><span class="cite-bracket" id="mwFw">[</span>5<span class="cite-bracket" id="mwGA">]</span></span></a></sup> (<span about="#mwt6" class="ext-phonos" data-mw='{"name":"phonos","attrs":{"file":"Nl-be guy verhofstadt.ogg"},"body":{"extsrc":"uitspraak"},"parts":[{"template":{"target":{"wt":"uitspraak","href":"./Sjabloon:Uitspraak"},"params":{"1":{"wt":"Nl-be guy verhofstadt.ogg"}},"i":0}}]}' id="mwGQ" typeof="mw:Extension/phonos mw:Transclusion"><span class="noexcerpt ext-phonos-PhonosButton oo-ui-widget oo-ui-widget-enabled oo-ui-buttonElement oo-ui-buttonElement-frameless oo-ui-buttonElement-size-medium oo-ui-iconElement oo-ui-labelElement oo-ui-buttonWidget" data-n

If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [ ]:
# <10 lines
url = "https://nl.wikipedia.org/wiki/Guy_Verhofstadt"
headers ={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
r = requests.get(url, headers = headers)
print(url, r.status_code) 
content = r.content
soup = BeautifulSoup(content, "html")

#for tag in soup.find_all("p"):
    #print(tag.text)

#for tag in soup.find_all("p"):
    #if len(tag.text.split()) > 8 and len(tag.text.strip()) > 30:
        #first_paragraph = tag.text
        #break

for p in paragraphs:
    if p.find("b"):
        first_paragraph = p.get_text()
        break
print(first_paragraph)


https://nl.wikipedia.org/wiki/Guy_Verhofstadt 403
Guy Maurice Marie-Louise Verhofstadt[5] (uitspraakⓘ) (Dendermonde, 11 april 1953) is een Belgisch politicus voor de Open Vlaamse Liberalen en Democraten (Open Vld). Hij was premier van België van 12 juli 1999 tot 20 maart 2008 in drie regeringen. Hij beëindigde zijn actieve politieke carrière in het Europees Parlement, waar hij van 2009 tot 2019 fractieleider van de Alliantie van Liberalen en Democraten voor Europa (ALDE) was.[6]


At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [ ]:
# 10 lines
# def get_first_paragraph(wikipedia_url):
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)
    headers ={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    
#   print(wikipedia_url) # keep this for the rest of the notebook
#   [insert your code]
    r = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(r.text, "html.parser")
    for p in soup.find_all("p"):
        if p.find("b"):
            first_paragraph = p.get_text()
#   return first_paragraph
            return first_paragraph





In [ ]:
# Test: 3 lines
result = get_first_paragraph("https://nl.wikipedia.org/wiki/Guy_Verhofstadt")
print(result)


https://nl.wikipedia.org/wiki/Guy_Verhofstadt
Guy Maurice Marie-Louise Verhofstadt[5] (uitspraakⓘ) (Dendermonde, 11 april 1953) is een Belgisch politicus voor de Open Vlaamse Liberalen en Democraten (Open Vld). Hij was premier van België van 12 juli 1999 tot 20 maart 2008 in drie regeringen. Hij beëindigde zijn actieve politieke carrière in het Europees Parlement, waar hij van 2009 tot 2019 fractieleider van de Alliantie van Liberalen en Democraten voor Europa (ALDE) was.[6]


### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [ ]:
# 3 lines
import re
cleaned = re.sub("\(.*?\)", "", first_paragraph)
cleaned = re.sub("\[.*?\]", "", first_paragraph)



<>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
<>:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
<>:4: SyntaxWarning: "\[" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\["? A raw string is also an option.
C:\Users\alec4\AppData\Local\Temp\ipykernel_27268\1793764652.py:3: SyntaxWarning: "\(" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\("? A raw string is also an option.
  cleaned = re.sub("\(.*?\)", "", first_paragraph)
C:\Users\alec4\AppData\Local\Temp\ipykernel_27268\1793764652.py:4: SyntaxWarning: "\[" is an invalid escape sequence. Such 

Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [ ]:
# 10 lines
import re
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)
    headers ={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    
#   print(wikipedia_url) # keep this for the rest of the notebook
#   [insert your code]
    r = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(r.text, "html.parser")
    for p in soup.find_all("p"):
        #print(p.find("b"))
        if p.find("b"):
            first_paragraph = p.get_text()
            first_paragraph = re.sub(r"\(.*?\)", "", first_paragraph)
            first_paragraph = re.sub(r"\[.*?\]", "", first_paragraph)
            first_paragraph = re.sub(" +", " ", first_paragraph)
            first_paragraph = re.sub(r"ⓘ", "", first_paragraph)
            first_paragraph = re.sub(r"Écouter", "", first_paragraph)
            first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph)
#   return first_paragraph
            return first_paragraph
        
result = get_first_paragraph("https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande")
print(result)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande , né le 12 août 1954 à Rouen , est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



In [ ]:

## Van Sooyoung 
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)
    req = requests.get(wikipedia_url, headers = {"User-Agent" : "Mozilla/5.0"})
    soup = BeautifulSoup(req.text, "html.parser")
    paragraphs = soup.find_all("p")
    first_paragraph = "" #add empty varaible for the safety
    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = paragraph.get_text()
            break
    
    if first_paragraph is None:
        for paragraph in paragraphs:
            if paragraph.get_text().strip():
                first_paragraph = paragraph.get_text()
                break


    #first_paragraph = re.sub(r"\[[^\]]*\]", "", first_paragraph)
    #first_paragraphirst_paragraph = re.sub(r"[[^]]]", "", first_paragraph)
    #first_paragraph = re.sub(r"\s+", " ", first_paragraph)
    first_paragraph = re.sub(r"[[^]]]/", "", first_paragraph)
    first_paragraph = re.sub(r"(/[^)]/\s[^)])", "", first_paragraph)
    first_paragraph = re.sub(r"ⓘ", "", first_paragraph)
    first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph)
    first_paragraph = re.sub(r"\s,", ",", first_paragraph)
    
    #first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph)
    return first_paragraph

result = get_first_paragraph("https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande")
print(result)


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouter, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



C:\Users\alec4\AppData\Local\Temp\ipykernel_27268\1469389086.py:23: FutureWarning: Possible nested set at position 1
  first_paragraph = re.sub(r"[[^]]]/", "", first_paragraph)


Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [ ]:
# < 20 lines


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
# < 20 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    countries_url = "https://country-leaders.onrender.com/countries"
    cookies = requests.get(cookie_url)
    countries = requests.get(countries_url, cookies = cookies.cookies).json()
    leaders_per_country = {}
    for country in countries:
        leaders_url = "https://country-leaders.onrender.com/leaders?country=" + str(country)
        leaders = requests.get(leaders_url, cookies = cookies.cookies).json()
        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"])
            leaders_per_country[country] = leaders
    return leaders_per_country

In [ ]:
# Check the output of your function (2 lines)
leaders = get_leaders()
print(leaders.keys)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

TypeError: string indices must be integers, not 'str'

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
# < 25 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    countries_url = "https://country-leaders.onrender.com/countries"
    cookies = requests.get(cookie_url)
    countries_res = requests.get(countries_url, cookies = cookies.cookies)

    if countries_res.status_code != 200:
        cookies = requests.get(cookie_url)
        countries_res = requests.get(countries_url, cookies = cookies.cookies).json()
    countries = countries_res.json()
    leaders_per_country = {}
    for country in countries:
        leaders_url = "https://country-leaders.onrender.com/leaders?country=" + str(country)
        leaders_res = requests.get(leaders_url, cookies = cookies.cookies)
        if leaders_res.status_code != 200:
            cookies = requests.get(cookie_url)
            leaders_res = requests.get(leaders_url, cookies = cookies.cookies)
        leaders = leaders_res.json()
        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"])
            leaders_per_country[country] = leaders
    return leaders_per_country


Check the output of your function again.

In [ ]:
# Check the output of your function (1 line)
leaders = get_leaders()
print(leaders)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [7]:
# < 20 lines   verkeerde versie !!!!!!! 
import re
import requests
from bs4 import BeautifulSoup

def get_first_paragraph(wikipedia_url, session):
    print(wikipedia_url)
    req = session.get(wikipedia_url, headers = {"User-Agent" : "Mozilla/5.0"})
    soup = BeautifulSoup(req.content, "html.parser")
    paragraphs = soup.find_all("p")
    first_paragraph = None #add empty varaible for the safety
    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = paragraph.get_text()
            break
    
    if first_paragraph is None:
        for paragraph in paragraphs:
            if paragraph.get_text().strip():
                first_paragraph = paragraph.get_text()
                break


    first_paragraph = p.get_text()
    first_paragraph = re.sub(r"\(.*?\)", "", first_paragraph)
    first_paragraph = re.sub(r"\[.*?\]", "", first_paragraph)
    first_paragraph = re.sub(" +", " ", first_paragraph)
    first_paragraph = re.sub(r"ⓘ", "", first_paragraph)
    first_paragraph = re.sub(r"Écouter", "", first_paragraph)
    first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph)
    
    return first_paragraph

In [16]:
#Juiste versie 

import re
import requests
from bs4 import BeautifulSoup


def get_first_paragraph(wikipedia_url, session):
    print(wikipedia_url)

    req = session.get(wikipedia_url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(req.content, "html.parser")

    paragraphs = soup.find_all("p")

    first_paragraph = None

    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = paragraph.get_text()
            break
    
    if first_paragraph is None:
            for paragraph in paragraphs:
                if paragraph.get_text().strip():
                    first_paragraph = paragraph.get_text()
                    break
    
    if first_paragraph is None:
         return ""


    # cleaning
    first_paragraph = re.sub(r"\(.*?\)", "", first_paragraph)
    first_paragraph = re.sub(r"\[.*?\]", "", first_paragraph)
    first_paragraph = re.sub(" +", " ", first_paragraph)
    first_paragraph = re.sub(r"ⓘ", "", first_paragraph)
    first_paragraph = re.sub(r"Écouter", "", first_paragraph)
    first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph)

    return first_paragraph


Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [17]:
# <25 lines

def get_leaders():
    session = requests.session()
    root_url = "https://country-leaders.onrender.com"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    countries_url = "https://country-leaders.onrender.com/countries"
    cookies = requests.get(cookie_url)
    countries_res = requests.get(countries_url, cookies = cookies.cookies)

    if countries_res.status_code != 200:
        cookies = requests.get(cookie_url)
        countries_res = requests.get(countries_url, cookies = cookies.cookies).json()
    countries = countries_res.json()
    leaders_per_country = {}
    for country in countries:
        leaders_url = "https://country-leaders.onrender.com/leaders?country=" + str(country)
        leaders_res = requests.get(leaders_url, cookies = cookies.cookies)
        if leaders_res.status_code != 200:
            cookies = requests.get(cookie_url)
            leaders_res = requests.get(leaders_url, cookies = cookies.cookies)
        leaders = leaders_res.json()
        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"],session)
            leaders_per_country[country] = leaders
    return leaders_per_country

Test your new functions.



In [18]:

leaders_per_country = get_leaders()
print(leaders_per_country)


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [ ]:
# 3 lines
import json
with open("leaders.json", "w", encoding = "UTF-8") as f:
    json.dump(leaders_per_country,f, ensure_ascii=False, indent=4)

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [ ]:
# 3 lines
with open("leaders.json", "r", encoding = "UTF-8") as f:
    data = json.load(f)
    


Make a function `save(leaders_per_country)` to call this code easily.

In [ ]:
# 3 lines
def save(leaders_per_county):
    with open("leaders.json", "w", encoding = "UTF-8") as f:
        json.dump(leaders.per_country, f, ensure_ascii=False, indent=4)

In [ ]:
# Call the function (1 line)
leaders_per_country = get_leaders()
save(leaders_per_country)


## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!